# Chapter 9 &mdash; State Elimination: Bypass Edges and the $m\times n$ Rule

**Concept 2 of the Chapter 9 decomposition:** *State Elimination: Bypass Edges, Self-Loops, and the $m\times n$ Rule*

Delete a state $s$ and replace each in&ndash;out pair by one edge labelled (incoming)(self-loop)$^*$(outgoing).

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-State-Elimination/Concept-State-Elimination.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The core rule. To delete state $s$, look at every **incoming** edge $p \to s$ and every
**outgoing** edge $s \to q$. For each such pair add a **bypass** edge

$$p \xrightarrow{\;R_{ps}\,(R_{ss})^*\,R_{sq}\;} q$$

where $R_{ss}$ is $s$'s **self-loop** (or $\varepsilon$ if there is none). If an edge
$p\to q$ already exists, **union** the new label onto it.

With $m$ incoming and $n$ outgoing edges this creates **$m\times n$** new edges &mdash;
the source of the blow-up in Concept 3.

The self-loop starred in the middle is the part beginners forget; it is what lets the
deleted state be visited any number of times.

## 2. Definitions

### The three label constructors Jove provides

In [ ]:
print("form_concat_RE(re1, re2) -- concatenation")
print("form_alt_RE([r1, r2, ...]) -- union")
print("form_kleene_RE(re)      -- star")
print("RE2Str(RE)              -- render a label as text")

### The bypass rule, written out

In [ ]:
def bypass_label(Rps, Rss, Rsq):
    """the label of the edge that replaces p -> s -> q"""
    mid = form_kleene_RE(Rss) if Rss is not None else None
    e = Rps
    if mid is not None: e = form_concat_RE(e, mid)
    return form_concat_RE(e, Rsq)

## 3. Tests

A state with a self-loop: the star is essential.

In [ ]:
N = md2mc('''NFA
I : 0 -> S
S : 1 -> S        !! the self-loop
S : 0 -> F
''')
Gn = mk_gnfa(N)
_, _, restr = del_gnfa_states(Gn)
print("RE :", restr)
D = min_dfa(nfa2dfa(re2nfa(restr)))
for s in ['00', '010', '0110', '01110', '0', '01']:
    print("   %-8r accepted? %s" % (s, accepts_dfa(D, s)))
assert accepts_dfa(D, '01110') and not accepts_dfa(D, '01')
print("\n0 1* 0 -- the starred self-loop is right there in the middle.")

**$m\times n$:** two in-edges and three out-edges make six bypass edges.

In [ ]:
def count_bypass(m, n): return m * n
for m, n in [(1,1), (2,3), (3,3), (4,5)]:
    print("m=%d incoming, n=%d outgoing -> %d new edges" % (m, n, count_bypass(m, n)))
assert count_bypass(2, 3) == 6

Existing edges are **unioned**, not overwritten.

In [ ]:
Two = md2mc('''NFA
I : 0 -> S
I : 1 -> F        !! a direct edge that already exists
S : 0 -> F
''')
_, _, r2 = del_gnfa_states(mk_gnfa(Two))
print("RE :", r2)
D = min_dfa(nfa2dfa(re2nfa(r2)))
assert accepts_dfa(D, '1') and accepts_dfa(D, '00')
print("both '1' (direct) and '00' (via S) are accepted -- the labels were unioned")

Deleting in a different **order** gives a different-looking but equivalent RE.

In [ ]:
N3 = md2mc('''NFA
I : 0 -> A
A : 1 -> B
B : 0 -> F
A : 0 -> A
''')
# DelList must be a permutation of ALL the original states; Real_I and
# Real_F are appended by del_gnfa_states itself.
G1 = mk_gnfa(N3); _, _, ra = del_gnfa_states(G1, DelList=['I', 'A', 'B', 'F'])
G2 = mk_gnfa(N3); _, _, rb = del_gnfa_states(G2, DelList=['F', 'B', 'A', 'I'])
print("delete I,A,B,F :", ra)
print("delete F,B,A,I :", rb)
Da, Db = min_dfa(nfa2dfa(re2nfa(ra))), min_dfa(nfa2dfa(re2nfa(rb)))
assert iso_dfa(Da, Db)
print("\ndifferent strings, same language :", iso_dfa(Da, Db))

## 4. Exercises


1. Delete a state with **no** self-loop by hand. What goes in the middle?
2. Why must the self-loop be starred rather than just concatenated?
3. Delete a state with 3 in-edges and 4 out-edges. How many labels do you write?

In [ ]:
# Your work for the exercises above.